In [1]:
import os
import sys
from pathlib import Path

# Ensure services/typo is on sys.path regardless of nbconvert CWD
_here = Path.cwd()
for _c in [_here, _here.parent, _here.parent.parent]:
    if (_c / "app" / "__init__.py").exists():
        if str(_c) not in sys.path:
            sys.path.insert(0, str(_c))
        break

from app.store.paths import repo_root

FIGURES_DIR = repo_root() / "docs" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

import matplotlib
import numpy as np
import pandas as pd

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import scipy.stats as stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler

np.random.seed(42)

print(f"Figures directory: {FIGURES_DIR}")
print("Setup complete.")


Figures directory: /Users/meetbhadra/Lume/docs/figures
Setup complete.


In [2]:
# ==========================================================================
# DATA PROVENANCE
# ==========================================================================
# Synthetic data: generated by app/eda/synthesize.py with seed=42.
# Generator creates 200 paired observations (same passage x user under two
# typographic conditions). Source: 30 passages in sample_corpus.jsonl,
# derived from public-domain Project Gutenberg text (license: public domain).
# Documented in: docs/data_sources.md
#
# Hypotheses (from PRD §B.1.8):
#   H1: Increased letter spacing raises reading speed (WPM) for dyslexic readers.
#   H2: Text features (syllable density, FK grade) predict adaptation benefit (reward).
#   H3: Interaction between user reading profile and adaptation type (user x arm ANOVA).
# ==========================================================================
print("Data provenance documented. See docs/data_sources.md")


Data provenance documented. See docs/data_sources.md


In [3]:
from app.eda.synthesize import generate_synthetic_events

df = generate_synthetic_events(n_pairs=100, seed=42)
# Only work with synthetic data (strict filter — never combine with real-user data)
df_syn = df[df["data_source"] == "synthetic"].copy()

print(f"Synthetic dataset: {len(df_syn)} rows")
print(f"Users: {df_syn['user_id'].nunique()}, Passages: {df_syn['text_hash'].nunique()}")
print(f"WPM range: {df_syn['wpm'].min():.0f} – {df_syn['wpm'].max():.0f}")
print(f"Reward range: {df_syn['reward'].min():.3f} – {df_syn['reward'].max():.3f}")
df_syn.head(4)


Synthetic dataset: 200 rows
Users: 10, Passages: 100
WPM range: 110 – 252
Reward range: 0.333 – 0.810


,user_id,text_hash,features_json,adaptation_config_json,arm_index,recommendation_source,was_user_modified,word_count,wpm,comprehension_score,comprehension_type,reward,data_source,has_letter_spacing,condition
0,synth_user_01,synth_sc_021_000,"{""avg_word_len"": 5.291415123926329, ""syllable_...","{""letter_spacing_em"": 0.0, ""word_spacing_em"": ...",0,demo_seed,False,75,155.0,0.650,self_rated,0.4890,synthetic,False,default
1,synth_user_01,synth_sc_021_000,"{""avg_word_len"": 5.291415123926329, ""syllable_...","{""letter_spacing_em"": 0.04, ""word_spacing_em"":...",15,demo_seed,False,75,172.8,0.691,self_rated,0.5510,synthetic,True,adapted
2,synth_user_03,synth_sc_009_001,"{""avg_word_len"": 4.294959652748713, ""syllable_...","{""letter_spacing_em"": 0.0, ""word_spacing_em"": ...",0,demo_seed,False,75,110.0,0.550,self_rated,0.3330,synthetic,False,default
3,synth_user_03,synth_sc_009_001,"{""avg_word_len"": 4.294959652748713, ""syllable_...","{""letter_spacing_em"": 0.04, ""word_spacing_em"":...",4,demo_seed,False,75,137.2,0.550,self_rated,0.4093,synthetic,True,adapted


In [4]:
import json

# Parse features_json into columns
feats = df_syn["features_json"].apply(json.loads).apply(pd.Series)
df_syn2 = pd.concat([df_syn.reset_index(drop=True), feats.reset_index(drop=True)], axis=1)

fig, axes = plt.subplots(2, 3, figsize=(12, 7))
axes = axes.flatten()
feature_cols = ["avg_word_len", "syllable_density", "freq_percentile_mean",
                "sentence_count", "flesch_kincaid", "wpm"]
feature_labels = ["Avg word length", "Syllable density", "Freq percentile mean",
                  "Sentence count", "Flesch-Kincaid grade", "WPM"]
for ax, col, label in zip(axes, feature_cols, feature_labels):
    ax.hist(df_syn2[col].dropna(), bins=20, color="#4a7fa5", edgecolor="white", alpha=0.85)
    ax.set_title(label, fontsize=11)
    ax.set_xlabel(label, fontsize=9)
    ax.set_ylabel("Count", fontsize=9)
    ax.tick_params(labelsize=8)

fig.suptitle("Feature Distributions [synthetic, N=200]", fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
out = FIGURES_DIR / "feature_distributions.png"
plt.savefig(out, dpi=120, bbox_inches="tight")
plt.close()
print(f"Saved: {out}")


Saved: /Users/meetbhadra/Lume/docs/figures/feature_distributions.png


In [5]:
num_cols = ["avg_word_len", "syllable_density", "freq_percentile_mean",
            "sentence_count", "flesch_kincaid", "wpm", "comprehension_score", "reward"]
corr = df_syn2[num_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(corr.values, cmap="coolwarm", vmin=-1, vmax=1, aspect="auto")
ax.set_xticks(range(len(num_cols)))
ax.set_yticks(range(len(num_cols)))
labels = ["avg_word_len", "syllable_dens", "freq_pct_mean",
          "sent_count", "fk_grade", "wpm", "comp_score", "reward"]
ax.set_xticklabels(labels, rotation=40, ha="right", fontsize=9)
ax.set_yticklabels(labels, fontsize=9)
for i in range(len(num_cols)):
    for j in range(len(num_cols)):
        ax.text(j, i, f"{corr.values[i, j]:.2f}", ha="center", va="center",
                fontsize=7, color="white" if abs(corr.values[i, j]) > 0.5 else "black")
plt.colorbar(im, ax=ax, shrink=0.8)
ax.set_title("Correlation Heatmap [synthetic, N=200]", fontsize=12, fontweight="bold")
plt.tight_layout()
out = FIGURES_DIR / "corr_heatmap.png"
plt.savefig(out, dpi=120, bbox_inches="tight")
plt.close()
print(f"Saved: {out}")


Saved: /Users/meetbhadra/Lume/docs/figures/corr_heatmap.png


In [6]:
# ==========================================================================
# H1: Paired t-test — letter spacing vs. WPM
# ==========================================================================
# Design: pivot on (user_id, text_hash) to create paired columns.
# Each row = same user reading the same passage with and without letter spacing.
# This preserves the pairing; joining on has_letter_spacing would destroy it.
# ==========================================================================

import json

# Separate letter-spacing vs. no-letter-spacing rows
df_ls = df_syn2[df_syn2["has_letter_spacing"] == True].copy()
df_no = df_syn2[df_syn2["has_letter_spacing"] == False].copy()

# Merge on (user_id, text_hash) — inner join to get paired observations only
paired = df_no[["user_id", "text_hash", "wpm"]].merge(
    df_ls[["user_id", "text_hash", "wpm"]],
    on=["user_id", "text_hash"],
    suffixes=("_default", "_letter_spacing"),
)

n_total = len(df_syn2)
n_pairs = len(paired)
pair_pct = n_pairs / (n_total // 2) * 100 if n_total > 0 else 0
print(f"Paired rows: {n_pairs} ({pair_pct:.0f}% of expected)")
assert n_pairs / (n_total // 2) >= 0.80, (
    f"H1: fewer than 80% of rows could be paired ({pair_pct:.0f}%) — "
    "check the synthetic generator"
)

wpm_default   = paired["wpm_default"].values
wpm_ls        = paired["wpm_letter_spacing"].values

t_stat, p_val = stats.ttest_rel(wpm_default, wpm_ls)
lift_wpm = (wpm_ls - wpm_default).mean()

ci_95 = stats.t.ppf(0.975, df=n_pairs - 1) * (wpm_ls - wpm_default).std() / np.sqrt(n_pairs)
mean_diff = wpm_ls.mean() - wpm_default.mean()

print(f"\nH1 Results:")
print(f"  Mean WPM (default):         {wpm_default.mean():.1f}")
print(f"  Mean WPM (letter spacing):  {wpm_ls.mean():.1f}")
print(f"  Mean lift:                  {lift_wpm:+.1f} WPM")
print(f"  t-statistic:                {t_stat:.3f}")
print(f"  p-value:                    {p_val:.4f}")
print(f"  95% CI for lift:            [{mean_diff - ci_95:.1f}, {mean_diff + ci_95:.1f}]")

# Figure: H1 confidence interval
fig, ax = plt.subplots(figsize=(8, 4))
conditions = ["Default", "Letter Spacing"]
means = [wpm_default.mean(), wpm_ls.mean()]
sds   = [wpm_default.std() / np.sqrt(n_pairs), wpm_ls.std() / np.sqrt(n_pairs)]
colors = ["#7b9ec0", "#2e6da4"]
bars = ax.bar(conditions, means, yerr=[sd * 1.96 for sd in sds],
              color=colors, capsize=8, width=0.5,
              error_kw={"linewidth": 1.5, "ecolor": "#333"})
ax.set_ylabel("Mean WPM", fontsize=11)
ax.set_title(
    f"H1: Letter Spacing → WPM [synthetic, N={n_pairs} pairs]\n"
    f"lift = {lift_wpm:+.1f} WPM  t={t_stat:.2f}  p={p_val:.3f}",
    fontsize=11, fontweight="bold",
)
ax.set_ylim(0, max(means) * 1.3)
for bar, mean in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width() / 2, mean + 5,
            f"{mean:.1f}", ha="center", va="bottom", fontsize=10)
plt.tight_layout()
out = FIGURES_DIR / "h1_spacing_ci.png"
plt.savefig(out, dpi=120, bbox_inches="tight")
plt.close()
print(f"Saved: {out}")

print("\nINTERPRETATION: ", end="")
if p_val < 0.05:
    print(f"Reject H0. Letter spacing associated with +{lift_wpm:.1f} WPM lift "
          f"(paired t-test, p={p_val:.3f}, synthetic data, N={n_pairs} pairs). "
          "Caution: synthetic effect is by construction; real-user validation needed.")
else:
    print(f"Fail to reject H0. No statistically significant WPM difference "
          f"(p={p_val:.3f}). Lift = {lift_wpm:.1f} WPM (synthetic).")


Paired rows: 100 (100% of expected)

H1 Results:
  Mean WPM (default):         151.8
  Mean WPM (letter spacing):  178.6
  Mean lift:                  +26.8 WPM
  t-statistic:                -24.324
  p-value:                    0.0000
  95% CI for lift:            [24.6, 29.0]
Saved: /Users/meetbhadra/Lume/docs/figures/h1_spacing_ci.png

INTERPRETATION: Reject H0. Letter spacing associated with +26.8 WPM lift (paired t-test, p=0.000, synthetic data, N=100 pairs). Caution: synthetic effect is by construction; real-user validation needed.


In [7]:
# ==========================================================================
# H2: OLS regression — text features predict ADAPTATION BENEFIT
# Outcome: reward_adapted - reward_default (per paired observation)
# Text complexity (FK grade, syllable density) moderates the benefit
# of letter spacing in the synthetic model.
# ==========================================================================

from app.ml.features import build_feature_vector
from app.schemas import AdaptationConfig, TextFeatures


def row_to_feature_vec(row):
    try:
        feats_dict = json.loads(row["features_json"]) if isinstance(row["features_json"], str) else row["features_json"]
        cfg_dict   = json.loads(row["adaptation_config_json"]) if isinstance(row["adaptation_config_json"], str) else row["adaptation_config_json"]
        tf  = TextFeatures(**feats_dict)
        cfg = AdaptationConfig(**cfg_dict)
        return build_feature_vector(tf, cfg)
    except Exception:
        return None

# Compute per-pair adaptation benefit: reward_adapted - reward_default
df_def = df_syn2[df_syn2["condition"] == "default"][["user_id","text_hash","reward","avg_word_len","syllable_density","freq_percentile_mean","sentence_count","flesch_kincaid"]]
df_adp = df_syn2[df_syn2["condition"] == "adapted"][["user_id","text_hash","reward"]]
paired_h2 = df_def.merge(df_adp, on=["user_id","text_hash"], suffixes=("_default","_adapted"))
paired_h2["adaptation_benefit"] = paired_h2["reward_adapted"] - paired_h2["reward_default"]
print(f"Paired for H2: {len(paired_h2)} rows")
print(f"Adaptation benefit: mean={paired_h2['adaptation_benefit'].mean():.3f}, sd={paired_h2['adaptation_benefit'].std():.3f}")

# OLS: adaptation_benefit ~ text features
formula = "adaptation_benefit ~ avg_word_len + syllable_density + freq_percentile_mean + sentence_count + flesch_kincaid"
ols_result = smf.ols(formula, data=paired_h2).fit()
print(ols_result.summary())

r2 = ols_result.rsquared
print(f"\nR² = {r2:.3f}")
print(f"\nINTERPRETATION: ", end="")
if r2 >= 0.30:
    print(f"Text features explain {r2*100:.1f}% of adaptation benefit variance (R²={r2:.3f}). "
          "FK grade and syllable density are significant predictors of letter-spacing benefit. "
          "[synthetic, N={len(paired_h2)}] — inflated R² expected on synthetic data.")
else:
    print(f"Text features explain {r2*100:.1f}% of adaptation benefit variance. "
          "Low R² suggests individual differences dominate over text complexity. [synthetic]")

# Figure: residuals
fitted = ols_result.fittedvalues
residuals = ols_result.resid

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(fitted, residuals, alpha=0.5, s=20, color="#4a7fa5")
axes[0].axhline(0, color="#cc4a4a", linewidth=1.2)
axes[0].set_xlabel("Fitted values", fontsize=10)
axes[0].set_ylabel("Residuals", fontsize=10)
axes[0].set_title(f"H2: Residuals vs Fitted [synthetic, N={len(paired_h2)}]", fontsize=10, fontweight="bold")
sm.qqplot(residuals, line="s", ax=axes[1], alpha=0.5, marker="o", markersize=3, color="#4a7fa5")
axes[1].set_title(f"H2: Q-Q Plot [synthetic, N={len(paired_h2)}]", fontsize=10, fontweight="bold")
plt.tight_layout()
out_res = FIGURES_DIR / "h2_residuals.png"
plt.savefig(out_res, dpi=120, bbox_inches="tight")
plt.close()
print(f"Saved: {out_res}")

fig2, ax2 = plt.subplots(figsize=(6, 5))
sm.qqplot(residuals, line="s", ax=ax2, alpha=0.5, marker="o", markersize=4, color="#4a7fa5")
ax2.set_title(f"H2: Q-Q Plot Residuals [synthetic, N={len(paired_h2)}]", fontsize=10, fontweight="bold")
plt.tight_layout()
out_qq = FIGURES_DIR / "h2_qq.png"
plt.savefig(out_qq, dpi=120, bbox_inches="tight")
plt.close()
print(f"Saved: {out_qq}")


Paired for H2: 100 rows
Adaptation benefit: mean=0.087, sd=0.043
                            OLS Regression Results                            
Dep. Variable:     adaptation_benefit   R-squared:                       0.562
Model:                            OLS   Adj. R-squared:                  0.539
Method:                 Least Squares   F-statistic:                     24.12
Date:                Sat, 09 May 2026   Prob (F-statistic):           1.53e-15
Time:                        13:26:13   Log-Likelihood:                 214.07
No. Observations:                 100   AIC:                            -416.1
Df Residuals:                      94   BIC:                            -400.5
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------

/Users/meetbhadra/Lume/services/typo/.venv/lib/python3.11/site-packages/statsmodels/graphics/gofplots.py:1041: UserWarning: color is redundantly defined by the 'color' keyword argument and the fmt string "b" (-> color=(0.0, 0.0, 1.0, 1)). The keyword argument will take precedence.
  ax.plot(x, y, fmt, **plot_style)


Saved: /Users/meetbhadra/Lume/docs/figures/h2_residuals.png
Saved: /Users/meetbhadra/Lume/docs/figures/h2_qq.png


In [8]:
# ==========================================================================
# H3: User × Adaptation ANOVA interaction
# NOTE: H3 is SYNTHETIC ONLY. Real-user N=10 cannot support this test.
# ==========================================================================
import statsmodels.formula.api as smf_ols

# Use 3+ users and 3 adaptation types for interpretable ANOVA
USERS_FOR_H3 = ["synth_user_00", "synth_user_01", "synth_user_02", "synth_user_03"]

def get_adapt_type(row):
    try:
        cfg = json.loads(row["adaptation_config_json"]) if isinstance(row["adaptation_config_json"], str) else row["adaptation_config_json"]
        cond = row.get("condition", "default")
        if cond == "default":
            return "default"
        elif cfg["letter_spacing_em"] > 0 and cfg["emphasis_on"]:
            return "spacing_emphasis"
        elif cfg["letter_spacing_em"] > 0:
            return "spacing"
        elif cfg["emphasis_on"]:
            return "emphasis"
        else:
            return "other"
    except Exception:
        return "other"

df_h3 = df_syn[df_syn["user_id"].isin(USERS_FOR_H3)].copy()
df_h3["adapt_type"] = df_h3.apply(get_adapt_type, axis=1)
# Focus on 3 interpretable categories: default, spacing, spacing_emphasis
df_h3 = df_h3[df_h3["adapt_type"].isin(["default", "spacing", "spacing_emphasis"])]

adapt_cats_h3 = ["default", "spacing", "spacing_emphasis"]
avail_cats_h3 = [a for a in adapt_cats_h3 if a in df_h3["adapt_type"].unique()]
cell_counts = df_h3.groupby(["user_id", "adapt_type"]).size()
missing_cells = [(u, a) for u in USERS_FOR_H3 for a in adapt_cats_h3
                 if (u, a) not in cell_counts.index or cell_counts[(u, a)] == 0]

n_h3 = len(df_h3)
interaction_p = 1.0

if not missing_cells:
    print(f"H3 dataset: {n_h3} rows, {df_h3['user_id'].nunique()} users, {df_h3['adapt_type'].nunique()} adapt types")
    print(f"Cell counts:\n{cell_counts.to_string()}")
    formula = "reward ~ C(user_id) + C(adapt_type) + C(user_id):C(adapt_type)"
    try:
        from statsmodels.stats.anova import anova_lm
        ols_model = smf_ols.ols(formula, data=df_h3).fit()
        anova_table = anova_lm(ols_model, typ=2)
        print("\nANOVA Table (Type II SS):")
        print(anova_table.round(4).to_string())
        interaction_key = "C(user_id):C(adapt_type)"
        interaction_p = anova_table.loc[interaction_key, "PR(>F)"] if interaction_key in anova_table.index else 1.0
        print(f"\nINTERPRETATION: User x Adaptation interaction p={interaction_p:.3f}. ", end="")
        if interaction_p < 0.05:
            print("Significant interaction: different users respond differently to adaptation types. "
                  "[H3 synthetic simulation, N=200, 4 synthetic users]")
        else:
            print("No significant interaction on synthetic data. "
                  "[H3 synthetic simulation, N=200, 4 synthetic users]")
    except Exception as exc:
        print(f"ANOVA failed: {exc}. [H3 synthetic simulation, N=200]")
else:
    print(f"INFO: Unbalanced cells ({len(missing_cells)} missing). ANOVA skipped honestly.")
    print("INTERPRETATION: H3 ANOVA skipped due to unbalanced design. "
          "Interaction plot shown with available categories. [H3 synthetic simulation, N=200]")

# Always produce the h3_interaction.png figure
fig, ax = plt.subplots(figsize=(9, 5))
for user in USERS_FOR_H3:
    means = [
        df_h3[(df_h3["user_id"]==user) & (df_h3["adapt_type"]==a)]["reward"].mean()
        if a in avail_cats_h3 and len(df_h3[(df_h3["user_id"]==user) & (df_h3["adapt_type"]==a)]) > 0
        else float("nan")
        for a in avail_cats_h3
    ]
    ax.plot(avail_cats_h3, means, marker="o", label=user, linewidth=1.8, markersize=7)
ax.set_xlabel("Adaptation type", fontsize=11)
ax.set_ylabel("Mean reward", fontsize=11)
ax.set_title(
    f"H3: User × Adaptation Interaction [H3 synthetic simulation, N={n_h3}, {len(USERS_FOR_H3)} synthetic users]",
    fontsize=10, fontweight="bold",
)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
out = FIGURES_DIR / "h3_interaction.png"
plt.savefig(out, dpi=120, bbox_inches="tight")
plt.close()
print(f"Saved: {out}")


H3 dataset: 98 rows, 4 users, 3 adapt types
Cell counts:
user_id        adapt_type      
synth_user_00  default             12
               spacing              9
               spacing_emphasis     3
synth_user_01  default             13
               spacing              7
               spacing_emphasis     6
synth_user_02  default             11
               spacing              5
               spacing_emphasis     6
synth_user_03  default             13
               spacing              7
               spacing_emphasis     6

ANOVA Table (Type II SS):
                          sum_sq    df         F  PR(>F)
C(user_id)                0.7534   3.0  251.1285  0.0000
C(adapt_type)             0.1824   2.0   91.2161  0.0000
C(user_id):C(adapt_type)  0.0035   6.0    0.5867  0.7401
Residual                  0.0860  86.0       NaN     NaN

INTERPRETATION: User x Adaptation interaction p=0.740. No significant interaction on synthetic data. [H3 synthetic simulation, N=200, 4 synthe

Saved: /Users/meetbhadra/Lume/docs/figures/h3_interaction.png


In [9]:
# ==========================================================================
# 5-fold Cross-Validation: Ridge regression reward prediction
# ==========================================================================

from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Build full 47-dim feature matrix
vecs_cv = df_syn.apply(row_to_feature_vec, axis=1)
valid_cv = vecs_cv.notna()
X_cv = np.stack(vecs_cv[valid_cv].values)
y_cv = df_syn.loc[valid_cv, "reward"].values

kf = KFold(n_splits=5, shuffle=True, random_state=42)
pipeline = Pipeline([("scaler", StandardScaler()), ("ridge", Ridge(alpha=1.0))])

y_pred_all = np.zeros_like(y_cv)
y_true_all = np.zeros_like(y_cv)
r2_folds = []
baseline_r2_folds = []

for fold_i, (train_idx, test_idx) in enumerate(kf.split(X_cv)):
    X_train, X_test = X_cv[train_idx], X_cv[test_idx]
    y_train, y_test = y_cv[train_idx], y_cv[test_idx]
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    y_pred_all[test_idx] = y_pred
    y_true_all[test_idx] = y_test
    fold_r2 = r2_score(y_test, y_pred)
    # Baseline: predict mean of training set
    baseline_pred = np.full_like(y_test, y_train.mean())
    baseline_r2 = r2_score(y_test, baseline_pred)
    r2_folds.append(fold_r2)
    baseline_r2_folds.append(baseline_r2)
    print(f"  Fold {fold_i+1}: R² = {fold_r2:.3f}  (baseline R² = {baseline_r2:.3f})")

mean_r2 = np.mean(r2_folds)
mean_baseline = np.mean(baseline_r2_folds)
print(f"\n5-fold CV mean R²:      {mean_r2:.3f}")
print(f"Baseline mean R²:        {mean_baseline:.3f}")

# Figure: predicted vs actual
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(y_true_all, y_pred_all, alpha=0.5, s=18, color="#4a7fa5", label="Predictions")
lims = [min(y_true_all.min(), y_pred_all.min()) - 0.02,
        max(y_true_all.max(), y_pred_all.max()) + 0.02]
ax.plot(lims, lims, "r--", linewidth=1.2, label="Perfect prediction")
ax.set_xlabel("Actual reward", fontsize=11)
ax.set_ylabel("Predicted reward", fontsize=11)
ax.set_title(
    f"Personalized R²={mean_r2:.1f} | Baseline R²={mean_baseline:.1f}\n"
    f"[synthetic, N={len(y_cv)}]",
    fontsize=11, fontweight="bold"
)
ax.legend(fontsize=10)
ax.set_xlim(lims); ax.set_ylim(lims)
plt.tight_layout()
out = FIGURES_DIR / "cv_pred_actual.png"
plt.savefig(out, dpi=120, bbox_inches="tight")
plt.close()
print(f"Saved: {out}")

print(f"\nINTERPRETATION: 5-fold CV mean R²={mean_r2:.3f}. "
      f"Personalized model explains {mean_r2*100:.1f}% of reward variance vs. "
      f"baseline {mean_baseline*100:.1f}%. [synthetic, N={len(y_cv)}] "
      "Note: hackathon simplification — a production system would calibrate per-user baselines.")


  Fold 1: R² = 0.127  (baseline R² = -0.003)
  Fold 2: R² = -0.222  (baseline R² = -0.093)
  Fold 3: R² = 0.075  (baseline R² = -0.009)
  Fold 4: R² = 0.114  (baseline R² = -0.024)
  Fold 5: R² = 0.271  (baseline R² = -0.062)

5-fold CV mean R²:      0.073
Baseline mean R²:        -0.038


Saved: /Users/meetbhadra/Lume/docs/figures/cv_pred_actual.png

INTERPRETATION: 5-fold CV mean R²=0.073. Personalized model explains 7.3% of reward variance vs. baseline -3.8%. [synthetic, N=200] Note: hackathon simplification — a production system would calibrate per-user baselines.


In [10]:
# ==========================================================================
# Optional: Stat-reviewer cell (Anthropic API)
# Only runs if ANTHROPIC_MODEL env var is set AND anthropic is installed.
# Sends only regression diagnostics (numbers + summaries), NEVER raw text.
# ==========================================================================
import os

_model = os.environ.get("ANTHROPIC_MODEL", "").strip()
if not _model:
    print("AI reviewer skipped (ANTHROPIC_MODEL not set).")
else:
    try:
        import anthropic
        _client = anthropic.Anthropic()
        _summary = (
            f"I ran a reading-adaptation EDA on {len(df_syn)} synthetic observations. "
            f"H1 (letter spacing → WPM): t={t_stat:.3f}, p={p_val:.4f}, lift={lift_wpm:.1f} WPM. "
            f"H2 (text features → reward): OLS R²={r2:.3f}. "
            f"H3 (user x adaptation): ANOVA interaction p=reported above. "
            f"5-fold CV Ridge: mean R²={mean_r2:.3f} vs baseline {mean_baseline:.3f}. "
            "Please review the statistical methodology for correctness and flag any concerns."
        )
        _resp = _client.messages.create(
            model=_model,
            max_tokens=512,
            messages=[{"role": "user", "content": _summary}],
        )
        print("=== AI Stat Review ===")
        print(_resp.content[0].text)
    except ImportError:
        print("AI reviewer skipped (anthropic package not installed).")
    except Exception as exc:
        print(f"AI reviewer skipped (error: {exc}).")


AI reviewer skipped (ANTHROPIC_MODEL not set).
